In [1]:
# imports

import math

import numpy as np
import pandas as pd
from bokeh.io import output_notebook
from omni_drive import (
    Angle,
    Clock,
    Distance,
    DriveVector,
    Kalman2d,
    OmniDrive,
    PoseRecorder,
    SensorRecorder,
    Time,
)
from omni_drive.live import LivePlot

output_notebook(hide_banner=True)


In [2]:
robot = OmniDrive()

## Część 1: Ruch i odometria

**Zadanie 1 – test dryfu odometrii**

<small>Sprawdźmy, jak odometria zachowuje się w praktyce. Wykonaj test dryfu kół, realizując przejazd po zamkniętej trasie kwadratu. Uzupełnij funkcję <code>drive_square</code>:</small>


In [3]:
async def drive_square():
    side_time = Time.s(1.5)
    pause_time = Time.s(0.5)
    speed = 0.5

    for heading in (0, 90, 180, 270):
        await robot.drive_for(side_time, DriveVector(velocity=speed, heading=Angle.deg(heading)))
        await robot.pause_for(pause_time)

robot.reset_pose()

for i in range(3):  # zwiększ liczbę przejść gdy uzupełnisz definicję drive_square
    await drive_square()
    print(f"Przejście nr {i + 1}: Odometria: {robot.pose}")


Przejście nr 1: Odometria: Pose(x=Distance(meters=0.0), y=Distance(meters=0.0), yaw=Angle(radians=0.0))
Przejście nr 2: Odometria: Pose(x=Distance(meters=0.0), y=Distance(meters=0.0), yaw=Angle(radians=0.0))
Przejście nr 3: Odometria: Pose(x=Distance(meters=0.0), y=Distance(meters=0.0), yaw=Angle(radians=0.0))


<div class="alert alert-block alert-info">
  <small><b>Pytanie:</b> Czy robot wrócił w miejsce początkowe? Jak daleko znalazł się od niego?</small>
</div>

### Położenie na podstawie LiDARu

<small>Jak już wiesz, oprócz trzech par czujników odległości zamontowanych co 90°, robot posiada również skaner laserowy LiDAR, który mierzy odległość do 680 punktów w polu widzenia 240°.</small>

<small>Będziemy korzystać z pomiarów LiDAR-u dokładnie w tych samych kierunkach, w które skierowane są pozostałe sensory. Biblioteka udostępnia w tym celu gotowe właściwości API, które wyznaczają średnią arytmetyczną z 5 punktów lasera w osi każdego z czujników:</small>

* <small>`robot.lidar.beam_1` – wiązka w lewo (+90°),</small>
* <small>`robot.lidar.beam_2` – wiązka na wprost (0°),</small>
* <small>`robot.lidar.beam_3` – wiązka w prawo (-90°).</small>

<small>Uśrednienie kilku sąsiednich punktów skutecznie redukuje szum pojedynczego pomiaru lasera.</small>

<small>Znając przesunięcie robota względem trzech punktów pomiarowych (ścian: lewej, przedniej i prawej) potrafimy z dużą dokładnością określić jego pozycję w osi Y oraz trochę mniej dokładnie w osi X:</small>

<div style="display: flex; gap: 20px; justify-content: center; align-items: flex-start; margin: 20px 0;">
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_start.svg" alt="Pozycja startowa" style="width: 100%; height: auto;" />
  </div>
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_moved.svg" alt="Po przesunięciu" style="width: 100%; height: auto;" />
  </div>
</div>

_Gdy robot przesunie się o $x$ do przodu, odległość do przedniej ściany zmaleje o $x$, a gdy przesunie się o $y$ w lewo, odległość do lewej ściany zmaleje o $y$, a do prawej wzrośnie o $y$. Położenie względem startu to zatem:_

$$
x = f_0 - f, \qquad y = \frac{(l_0 - l) - (r_0 - r)}{2},
$$

<small>gdzie:</small>

* $f_0$, $l_0$, $r_0$ <small>to odległości do przedniej, lewej i prawej ściany zmierzone na starcie (przed ruszeniem robota),</small>
* $f$, $l$, $r$ <small>to aktualne odległości odczytywane w trakcie jazdy z wiązek LiDAR-u (odpowiednio: `beam_2`, `beam_1`, `beam_3`).</small>


<div class="alert alert-block alert-warning">
  <small>
  <b>Uwaga:</b> Takie położenie nie dryfuje, ponieważ ściany się nie przesuwają, ale uproszczenie to działa tylko w znanym, prostokątnym otoczeniu i przy stałej orientacji robota. W praktyce roboty zwykle poruszają się w dowolnym otoczeniu – budują wtedy z kolejnych skanów mapę i dopasowują do niej każdy nowy skan (scan matching); zasada pozostaje ta sama, ale ściany i punkty odniesienia są wyznaczane bezpośrednio z danych.
  </small>
</div>


**Zadanie 2 - odometria czy LiDAR**

<small>Wykorzystamy funkcję `drive_square` z poprzedniego ćwiczenia. Wstaw robota do prostokątnego kartonu i dostosuj długość przejazdu, tak, żeby poruszał się w środku nie obijając się o ściany.</small>

<small>Wykonamy trzy przejazdy, będziemy analizować pozycję robota raportowaną przez odometrię oraz obliczaną na podstawie danych z LiDAR-u.</small>

<small>Zaimplementuj logikę pozwalającą wyznaczać pozycję x oraz y za pomocą LiDARu.</small>


In [4]:
robot.reset_pose()

f0 = robot.lidar.beam_2
l0 = robot.lidar.beam_1
r0 = robot.lidar.beam_3

def lidar_get_x():
    f = robot.lidar.beam_2
    dx = f0.meters - f.meters
    return Distance.m(dx)

def lidar_get_y():
    l = robot.lidar.beam_1
    r = robot.lidar.beam_3
    dy = ((l0.meters - l.meters) - (r0.meters - r.meters)) / 2
    return Distance.m(dy)


In [5]:
robot.reset_pose()

rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria vs LiDAR"
)

live = LivePlot(rec).show()

def print_pos(label: str):
    pose = robot.pose
    px, py = pose.x.meters, pose.y.meters
    lx, ly = lidar_get_x().meters, lidar_get_y().meters

    d = math.hypot(px - lx, py - ly)
    print(f"{label}: Odometria: ({px:.3f}, {py:.3f}), LiDAR: ({lx:.3f}, {ly:.3f}), Różnica: {d:.3f} m")
    return d

print_pos("Pozycja startowa")

final_drift = 0.0
for i in range(3):
    await drive_square()

    # Sprawdzamy wartości oraz różnicę po każdym przejściu pętli
    final_drift = print_pos(f"Przejście nr {i + 1}")

live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'762f2fca-e5e8-459a-8800-6ef47fe64ad9': {…

Pozycja startowa: Odometria: (0.000, 0.000), LiDAR: (nan, nan), Różnica: nan m
Przejście nr 1: Odometria: (0.000, 0.000), LiDAR: (nan, nan), Różnica: nan m
Przejście nr 2: Odometria: (0.000, 0.000), LiDAR: (nan, nan), Różnica: nan m
Przejście nr 3: Odometria: (0.000, 0.000), LiDAR: (nan, nan), Różnica: nan m


<div class="alert alert-block alert-info">
  <small><b>Pytania:</b> Co jest dokładniejsze: odometria czy pozycja obliczona na podstawie LiDAR-u? Jak duża jest różnica? Dlaczego na zamkniętej trasie część błędu odometrii się znosi?</small>
</div>

<div class="alert alert-block alert-success">
  <small><b>Wskazówka:</b> Pozycja z LiDAR-u to nasz niezależny punkt odniesienia. Nie używamy jej do obliczeń &ndash; służy wyłącznie na wykresach do oceny błędu odometrii i skuteczności kolejnych algorytmów.</small>
</div>

**Czym jest szum procesu $Q$?**

<small>Zaobserwowana w teście różnica między odometrią a LiDAR-em na mecie (<code>final_drift</code>) to wynik kumulacji drobnych poślizgów kół na każdym kroku pętli sterowania. Pojedynczy błąd na jeden krok zegara jest rzędu ułamka milimetra, ale w miarę upływu czasu błędy sumują się, powodując narastający dryf pozycji robota.</small>

<small>W modelu ruchu robota tę losową niepewność opisujemy jako <b>wariancję szumu procesu</b> </small>$Q$<small> przypadającą na pojedynczy krok zegara. Skoro wariancje niezależnych błędów sumują się z każdym krokiem, po </small>$N$<small> próbkach zarejestrowanych przez <code>rec</code> całkowita wariancja dryfu wynosi:</small>

$$\sigma^2_{\text{całk}} \approx N \cdot Q \implies Q \approx \frac{\Delta_{\text{dryf}}^2}{N}$$

<small>Wykorzystaj zarejestrowaną liczbę kroków </small>$N$<small> z obiektu <code>rec.data</code> oraz końcowy dryf <code>final_drift</code>, aby obliczyć wariancję szumu procesu </small>$Q$<small> na pojedynczy krok:</small>


In [6]:
N_steps = len(rec.data["odometry"])

Q = (final_drift ** 2) / N_steps

print(f"Liczba próbek z rec:            N = {N_steps}")
print(f"Zmierzony dryf na mecie:       Δ = {final_drift * 1000:.1f} mm")
print(f"Oszacowany szum procesu:      √Q = {math.sqrt(Q) * 1000:.2f} mm / krok")
print(f"Wariancja szumu procesu:       Q = {Q:.6f} m²  ({Q * 1_000_000:.2f} mm²)")


Liczba próbek z rec:            N = 481
Zmierzony dryf na mecie:       Δ = nan mm
Oszacowany szum procesu:      √Q = nan mm / krok
Wariancja szumu procesu:       Q = nan m²


## Część 2: Czujniki odległości i Filtr Kalmana

**Zadanie 3 – Ruch wahadłowy**

<small>Skupimy się teraz na pozostałych czujnikach, zobaczymy jak radzą sobie w porównaniu do LiDARu. Użyjemy klasy `SensorRecorder`, aby zbierać odczyty czujników na jednym wspólnym panelu w trakcie trwania ruchu.

Twoim zadaniem jest zaimplementowanie płynnego ruchu wahadłowego:</small>

* <small>robot powinien poruszać się płynnym ruchem harmonicznym wahadła wokół pozycji początkowej,</small>
* <small>sterowanie prędkością powinno być funkcją okresową cosinus z prędkością <code>speed = 0.3</code> (np. <code>period = 3.0</code> s), co eliminuje szarpanie i poślizg kół na nawrotach,</small>
* <small>powtórz ruch przez 10 sekund (ok. 3 pełne cykle).</small>

In [7]:
robot.reset_pose()

clock = Clock()

rec = SensorRecorder(
    clock,
    title="Ruch wahadłowy: LiDAR vs US vs IR",
    styles={"ultrasonic": "scatter", "ir": "scatter", "lidar": "dashed"},
    lidar=lambda: robot.lidar.beam_2.meters,
    ultrasonic=lambda: robot.us_2.distance.meters,
    ir=lambda: robot.ir_2.distance.meters,
)

live = LivePlot(rec).show()

speed = 0.3
period = 3.0  # czas jednego cyklu wahadła [s]

try:
    async for t in clock:
        v = speed * math.cos(2 * math.pi * t.seconds / period)
        robot.set_vector(DriveVector(velocity=v, heading=Angle.deg(0)))

        if t > Time.s(10.0):
            break
finally:
    robot.stop()

live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'0488036c-8d3c-4560-ba63-a27719366dbb': {…

<div class="alert alert-block alert-info">
  <small><b>Prześledź wykresy:</b> Co na nich widać? Który z odczytów jest najdokładniejszy, a który najmniej dokladny?</small>
</div>

**Czym jest szum pomiaru $R$?**

<small>Nagraj teraz odczyty czujników, gdy robot stoi nieruchomo w miejscu przez 5 sekund. Sprawdźmy, jak szumią czujniki, kiedy robot nie jeździ.</small>


In [8]:
PAUSE_TIME = 5

rec_still = SensorRecorder(
    window=PAUSE_TIME,
    title="Szum w spoczynku",
    styles={"ultrasonic": "scatter", "ir": "scatter", "lidar": "scatter"},
    lidar=lambda: robot.lidar.beam_2.meters,
    ultrasonic=lambda: robot.us_2.distance.meters,
    ir=lambda: robot.ir_2.distance.meters,
)

live = LivePlot(rec_still).show()
await robot.pause_for(Time.s(PAUSE_TIME))
live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'4787d759-9ba7-4842-823c-af5aa51960c5': {…

In [9]:
df = pd.DataFrame(rec_still.data)
df.head()

,t,lidar,ultrasonic,ir
0,0.000000,NaN,NaN,NaN
1,0.050609,NaN,NaN,NaN
2,0.101158,NaN,NaN,NaN
3,0.151020,NaN,NaN,NaN
4,0.200399,NaN,NaN,NaN


<small>Każdy czujnik jest obarczony losową niedoskonałością aparatury, którą opisujemy statystycznie jako <b>wariancję szumu pomiaru</b> </small>$R$<small>. Skoro robot stał w miejscu, cała obserwowana zmienność odczytów wynika wyłącznie z szumów czujników.</small>

<small>Zamiast zwykłej wariancji (<code>np.var()</code>), która jest bezbronna wobec pojedynczych fałszywych ech czujnika ultradźwiękowego (outlierów), stosujemy odporną estymację <b>MAD</b> (odchylenie bezwzględne od mediany):</small>

$$\text{MAD} = \operatorname{median}(|z - \operatorname{median}(z)|), \qquad \sigma \approx 1{,}4826 \cdot \text{MAD}, \qquad R = \sigma^2$$

<small>Wyznaczone w ten sposób wartości </small>$R_{\text{ir}}$<small> oraz </small>$R_{\text{us}}$<small> posłużą jako wagi określające zaufanie do poszczególnych sensorów.</small>

In [10]:
def calc_R(series):
    clean = series.dropna()
    mad = float(np.median(np.abs(clean - np.median(clean))))
    return max((1.4826 * mad) ** 2, 1e-6)


In [11]:
R_ir = calc_R(df["ir"])
R_us = calc_R(df["ultrasonic"])

print("\n--- Odporna estymacja szumu losowego R (MAD) ---")
print(f"IR:         √R = {math.sqrt(R_ir) * 1000:5.1f} mm")
print(f"Ultrasonic:  √R = {math.sqrt(R_us) * 1000:5.1f} mm")



--- Odporna estymacja szumu losowego R (MAD) ---
LiDAR:       √R =   nan mm
IR:         √R =   nan mm
Ultrasonic:  √R =   nan mm


/home/radek/omni-drive-public/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/radek/omni-drive-public/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


**<small>Co by było, gdybyśmy użyli zwykłej wariancji zamiast odpornej miary MAD?</small>**

In [12]:
var_us = df["ultrasonic"].var()

print("--- Wpływ outlierów na czujnik ultradźwiękowy ---")
print(f"Zwykłe odchylenie std (np.var): √Var = {math.sqrt(var_us) * 1000:5.1f} mm")
print(f"Odporne odchylenie std (MAD):   √R   = {math.sqrt(R_us) * 1000:5.1f} mm")


--- Wpływ outlierów na czujnik ultradźwiękowy ---
Zwykłe odchylenie std (np.var): √Var =   nan mm
Odporne odchylenie std (MAD):   √R   =   nan mm


### Filtr Kalmana

<small>Odometria jest płynna, lecz <b>dryfuje</b> (błąd akumuluje się w czasie). Czujniki odległości (ultradźwiękowy, IR) nie dryfują, ale są <b>zaszumione</b>.</small>

<small><b>Filtr Kalmana</b> łączy oba źródła, wyznaczając optymalny <b>stan robota</b> </small>$x$<small>, wykonując naprzemiennie dwa kroki:</small>

---

#### 1. Krok predykcji (ruch robota z odometrii)
<small>Przesuwamy stan robota o przemieszczenie z kół </small>$\Delta x$<small>. Niepewność stanu (wariancja </small>$P$<small>) rośnie o szum procesu </small>$Q$<small>:</small>
$$x^- = x + \Delta x, \qquad P^- = P + Q$$

#### 2. Krok korekty (pomiar pozycji $z$)
<small>Pomiar </small>$z$<small> musi opisywać tę samą wielkość co stan </small>$x$<small>. Ze zmierzonego dystansu do ściany </small>$d$<small> wyznaczamy położenie robota jako zbliżenie względem startu (</small>$d_0$<small>):</small>
$$z = d_0 - d$$

<small>Obliczamy innowację (błąd prognozy) </small>$y$<small> oraz <b>wzmocnienie Kalmana</b> </small>$K \in [0, 1]$<small>, które określa stopień zaufania do pomiaru:</small>
$$y = z - x^-, \qquad K = \frac{P^-}{P^- + R}$$
<small>Korygujemy stan robota oraz redukujemy jego niepewność:</small>
$$x = x^- + K \cdot y, \qquad P = (1 - K) \cdot P^-$$

---

<small><b>Intuicja wzmocnienia</b> </small>($K$):<br>

* <small>Gdy sensor mocno szumi </small>$(R \gg P^-) \implies K \to 0$<small>: ignorujemy pomiar, polegamy na odometrii </small>$(x \approx x^-)$<small>.</small>
* <small>Gdy sensor jest bardzo precyzyjny </small>$(R \ll P^-) \implies K \to 1$<small>: ufamy pomiarowi </small>$(x \approx z)$<small>.</small>
* <small>Każda poprawna korekta zmniejsza niepewność stanu: </small>$(1 - K) \le 1 \implies P < P^-$<small>.</small>


**Zadanie 4.1 – Krok korekty Filtru Kalmana (1D)**

<small>Uzupełnij implementację kroku korekty Filtru Kalmana dla pozycji robota </small>$x$<small> na podstawie wzorów powyżej:</small>

In [13]:
def kalman_update(x, P, z, R):
    """
    Krok korekty (update) Filtru Kalmana 1D dla pozycji robota.
    x: stan a priori robota
    P: wariancja niepewności stanu a priori
    z: pomiar pozycji z czujnika
    R: wariancja szumu pomiarowego
    """
    y = z - x
    K = P / (P + R)
    x_new = x + K * y
    P_new = (1 - K) * P
    return x_new, P_new


**Zadanie 4.2 – Filtr Kalmana w ruchu wahadłowym (1D)**

<small>Połącz ruch wahadłowy z Zadania 3 z funkcją <code>kalman_update</code>: w każdym kroku pętli wykonaj predykcję z odometrii oraz korektę stanu czujnikiem ultradźwiękowym.</small>


In [14]:
robot.reset_pose()
clock = Clock()

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
d0_us = robot.us_2.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
d0_lidar = robot.lidar.beam_2.meters
rec = SensorRecorder(
    clock,
    title="Filtr Kalmana 1D (US) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"ultrasonic": "scatter", "kalman": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    ultrasonic=lambda: d0_us - robot.us_2.distance.meters,
    kalman=lambda: x,
    lidar=lambda: d0_lidar - robot.lidar.beam_2.meters,
)
live = LivePlot(rec).show()

speed = 0.3
period = 3.0

try:
    async for t in clock:
        # Ruch wahadłowy z Zadania 3:
        v = speed * math.cos(2 * math.pi * t.seconds / period)
        robot.set_vector(DriveVector(velocity=v, heading=Angle.deg(0)))

        # Predykcja:
        curr_odom_x = robot.pose.x.meters
        dx = curr_odom_x - last_odom_x
        last_odom_x = curr_odom_x
        x += dx
        P += Q

        # Pomiar:
        z = d0_us - robot.us_2.distance.meters

        # Korekta:
        x, P = kalman_update(x, P, z, R_us)

        if t > Time.s(10.0):
            break
finally:
    robot.stop()

live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'bb9e668c-2645-4bf8-bcc1-8daa5ca18ef7': {…

### Fuzja wielu czujników – w uproszczeniu

<small>Napisany przez nas algorytm Filtru Kalmana pozwala uwzględnić tylko jedno źródło. Z reguły tych źródeł będziemy posiadać wiele – w końcu nasz robot posiada aż sześć czujników oraz skaner laserowy.</small>

<small>Natomiast funkcja w takiej postaci bardzo dobrze nadaje się do nauki, a dołożenie kolejnego czujnika jest tak proste, jak wykonanie sekwencyjnie kolejnych aktualizacji Filtrem Kalmana:</small>

```python
x, P = kalman_update(x, P, z_us, R_us) # Filtr Kalmana z wejściem US
x, P = kalman_update(x, P, z_ir, R_ir) # Filtr Kalmana z wejściem US oraz IR
```

$$\text{Predykcja}(Q) \;\longrightarrow\; \text{Korekta}(z_{\text{US}}, R_{\text{US}}) \;\longrightarrow\; \text{Korekta}(z_{\text{IR}}, R_{\text{IR}})$$


**Zadanie 4.3 – Sekwencyjna fuzja czujników (US + IR)**

<small>Skopiuj kod z Zadania 4.2 i dodaj sekwencyjne wywołanie <code>kalman_update</code> dla sensora IR nałożone na US.</small>

<small>Na wspólnym wykresie zestawiamy: odometrię, stan po US (<code>kalman_us</code>), fuzję obu czujników (<code>kalman_fusion</code>) oraz LiDAR jako niezależny punkt odniesienia.</small>


In [15]:
robot.reset_pose()
clock = Clock()

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
d0_us = robot.us_2.distance.meters
d0_ir = robot.ir_2.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
x_us = 0.0
x_fusion = 0.0
d0_lidar = robot.lidar.beam_2.meters
rec = SensorRecorder(
    clock,
    title="Sekwencyjna fuzja (US + IR) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"kalman_us": "line", "kalman_fusion": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    kalman_us=lambda: x_us,
    kalman_fusion=lambda: x_fusion,
    lidar=lambda: d0_lidar - robot.lidar.beam_2.meters,
)
live = LivePlot(rec).show()

speed = 0.3
period = 3.0

try:
    async for t in clock:
        # Ruch wahadłowy i predykcja z Zadania 4.2:
        v = speed * math.cos(2 * math.pi * t.seconds / period)
        robot.set_vector(DriveVector(velocity=v, heading=Angle.deg(0)))

        curr_odom_x = robot.pose.x.meters
        dx = curr_odom_x - last_odom_x
        last_odom_x = curr_odom_x
        x += dx
        P += Q

        # Pomiary:
        z_us = d0_us - robot.us_2.distance.meters
        z_ir = d0_ir - robot.ir_2.distance.meters

        # Sekwencyjna korekta:
        x, P = kalman_update(x, P, z_us, R_us)
        x_us = x

        x, P = kalman_update(x, P, z_ir, R_ir)
        x_fusion = x

        if t > Time.s(10.0):
            break
finally:
    robot.stop()

live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'bca7f6d5-7ac7-4cf0-8183-bb855712012b': {…

<div class="alert alert-block alert-info">
  <small><b>Pytanie do przemyślenia:</b> Co gdyby czujnik był zamontowany <b>pod kątem</b>, a nie wprost w kierunku ściany? Czy wówczas również moglibyśmy użyć odczytu bezpośrednio (<i>z</i> &minus; <i>x</i>), czy filtr potrzebuje wiedzieć, w jakiej relacji do osi ruchu znajduje się dany sensor? Odpowiedź na to pytanie znajdziesz poniżej.</small>
</div>


### Pełnoprawna fuzja sensorów (Kalman 2D)

<small>Do tej pory stosowaliśmy uproszczenie w 1D. Pełnoprawny Filtr Kalmana radzi sobie z fuzją wielu źródeł znacznie wydajniej – operuje wprost na rachunku macierzowym, korygując cały stan robota w jednym kroku obliczeniowym.</small>

<small>Bezpośrednim wejściem do algorytmu są macierze, których wymiary wynikają z rozmiaru wektora stanu (</small>$n = 2$<small>) oraz liczby sygnałów pomiarowych (</small>$m = 6$<small>):</small>

* **Stan robota** $\mathbf{x} = \begin{bmatrix} x \\ y \end{bmatrix}$ <small>&ndash; współrzędne pozycji 2D robota na arenie (wymiar </small>$n = 2$<small>).</small>
* **Szum procesu** $\mathbf{Q}_{2 \times 2} = \operatorname{diag}([Q, Q])$ <small>&ndash; niepewność ruchu z odometrii na krok zegara (znamy z Zadania 2).</small>
* **Szum pomiaru** $\mathbf{R}_{6 \times 6} = \operatorname{diag}([R_{\text{ir}}, \dots, R_{\text{us}}, \dots])$ <small>&ndash; szumy pomiarowe 6 sensorów (3x IR, następnie 3x US).</small>
* **Macierz obserwacji** $\mathbf{H}_{6 \times 2}$ <small>&ndash; określa, <b>w jaki sposób każdy sensor wpływa na stan</b> (</small>$\mathbf{z} = \mathbf{H}\mathbf{x} + \mathbf{v}$<small>). Każdy wiersz reprezentuje jedno źródło pomiaru, a kolumny przypisują sensor do mierzonej współrzędnej (</small>$x$<small> lub </small>$y$<small>):</small>

$$\mathbf{H} = \left. \begin{bmatrix} 
1 & 0 \\
\vdots & \vdots 
\end{bmatrix} \right\} \text{6 wierszy (czujników)}
\quad
\begin{matrix} 
\leftarrow \text{IR przód (mierzy współrzędną } x) \\
\dots 
\end{matrix}$$

<small>Skaner LiDAR nie bierze udziału w filtracji – posłuży na wykresie jako niezależny punkt odniesienia.</small>


**Zadanie 5 – Fuzja odometrii i 6 czujników pokładowych**

<small>Dokończ definicję wektora źródeł pomiarowych </small>`z`<small> (odczyty znormalizowane o odległość bazową </small>$d_0$<small>) oraz uzupełnij macierze </small>`Q_mat`<small>, </small>`R_mat`<small> i </small>`H`<small>, a następnie utwórz obiekt filtru </small>`Kalman2d`<small>:</small>


In [16]:
robot.reset_pose()

d0_ir_f, d0_ir_l, d0_ir_r = robot.ir_2.distance.meters, robot.ir_1.distance.meters, robot.ir_3.distance.meters
d0_us_f, d0_us_l, d0_us_r = robot.us_2.distance.meters, robot.us_1.distance.meters, robot.us_3.distance.meters

z = [
    lambda: d0_ir_f - robot.ir_2.distance.meters,  # IR przód
    lambda: d0_ir_l - robot.ir_1.distance.meters,  # IR lewy
    lambda: d0_ir_r - robot.ir_3.distance.meters,  # IR prawy
    lambda: d0_us_f - robot.us_2.distance.meters,  # US przód
    lambda: d0_us_l - robot.us_1.distance.meters,  # US lewy
    lambda: d0_us_r - robot.us_3.distance.meters,  # US prawy
]

Q_mat = np.diag([Q, Q])

R_mat = np.diag([R_ir, R_ir, R_ir, R_us, R_us, R_us])

H = np.array([
    [1, 0],   # IR przód
    [0, 1],   # IR lewy
    [0, -1],  # IR prawy
    [1, 0],   # US przód
    [0, 1],   # US lewy
    [0, -1],  # US prawy
])

kalman = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)


<small>Uruchamiamy przejazd testowy na trasie kwadratu. Rejestrujemy jednocześnie na jednym wykresie dryfującą odometrię, stan robota z Filtru Kalmana 2D (fuzja 6 czujników) oraz niezależny punkt odniesienia z LiDAR-u:</small>


In [17]:
robot.reset_pose()
rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    kalman=lambda: kalman.state,
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria vs Filtr Kalmana 2D (6 sensorów) vs LiDAR (punkt odniesienia)"
)
live = LivePlot(rec).show()
for i in range(3):
    await drive_square()
live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'c915b789-96d0-4370-9d69-f4109a2b7e0d': {…

### <small>Co się dzieje przy złym strojeniu parametrów?</small>

<small>Wzmocnienie Kalmana </small>$K$<small> zależy od stosunku niepewności ruchu </small>$Q$<small> do szumu pomiaru </small>$R$<small>:</small>
* <small><b>Za duże </b></small>$R$<small><b> (lub za małe </b></small>$Q$<small><b>):</b> filtr nie ufa pomiarom, polega na odometrii i przejmuje jej narastający dryf.</small>
* <small><b>Za małe </b></small>$R$<small><b> (lub za duże </b></small>$Q$<small><b>):</b> filtr ufa bezkrytycznie każdemu odczytowi &rarr; stan robota drży od szumu aparatury.</small>

<small>Uruchamiamy trzy filtry 2D jednocześnie podczas jednego przejazdu po kwadracie:</small>
* <small><code>kf_optimal</code> &ndash; dobrze dobrane parametry (stabilny, domknięty kwadrat),</small>
* <small><code>kf_trust_odom</code> &ndash; </small>$R \times 1000$<small> (filtr ignoruje czujniki i dryfuje z kołami),</small>
* <small><code>kf_trust_sensor</code> &ndash; </small>$R \times 0{,}001$<small> (filtr jest nerwowy i szarpie pozycją przy każdym szumie czujników).</small>


In [18]:
kf_optimal      = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)
kf_trust_odom   = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat * 1000, H=H)
kf_trust_sensor = Kalman2d(robot, z=z, Q=Q_mat * 1000, R=R_mat * 0.001, H=H)

robot.reset_pose()

rec_tuning_2d = PoseRecorder(
    optimal=lambda: kf_optimal.state,
    trust_odom=lambda: kf_trust_odom.state,
    trust_sensor=lambda: kf_trust_sensor.state,
    title="Wpływ strojenia Q i R w 2D"
)

live = LivePlot(rec_tuning_2d).show()

for i in range(3):
    await drive_square()

live.stop()


BokehModel(layout=Layout(width='100%'), render_bundle={'docs_json': {'e9a9e3af-6acd-4ef6-8101-a1c4d08dad6f': {…